# HybridGuard — Revision Addons

**Goal.** Produce two pieces of statistical reporting the manuscript currently flags as queued:

* **A1 — Bootstrap 95% CIs** for Table 9 (universal canonicalization recovery), 1000 resamples per detector × condition.
* **A2 — Standard confidence-based ECE** (Guo et al. 2017, $\max(p, 1{-}p)$ binning) for Table 5.

Both need per-sample probability outputs, which are not in the public reproducibility extract — that's why this notebook re-runs the relevant evaluations from scratch.

**Hardware.** Colab Pro+ with an A100 GPU. Wall-clock ≈ 35–45 min.

**Workflow.**
1. Run cells top to bottom. (You can also `Runtime → Run all` and walk away — Pro+ background execution is supported.)
2. The final cell zips all outputs, copies the zip into Drive, and triggers a browser download. **The Drive copy is the canonical persistent backup**; the browser download is a convenience that may miss if you close the tab early.
3. Drag the zip into the OneDrive HybridGuard root and unzip. The zip preserves the manuscript-aligned subfolder layout (`paper/paper_v2_extract/...`).
4. Re-launch the dashboard (`cd dashboard && python3 dashboard.py`); the new "Universal Defense" tab and the "Standard ECE" calibration row pick up the new files automatically.

**Notes for n drift.** This notebook does **not** pin the test-set size to a specific n. The xTRam1 snapshot has changed since the manuscript's first run (was n=1625, currently n=2060). The notebook adapts to whatever n the dataset ships and prints a warning so you remember to update the manuscript caption.

## 1. Setup

In [ ]:
# 1.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.2 — Clone repo, add `src/` to sys.path, install runtime deps
#
# Why sys.path instead of `pip install -e .`:
#   The repo currently has both an empty stale `hybridguard/` directory at the root
#   and the real package at `src/hybridguard/`. Editable pip install picks the wrong
#   one. Adding `src/` to sys.path bypasses pip's package discovery entirely.
#
# Why no scikit-learn pin:
#   The version we used during the manuscript run was 1.5.x; Colab's preinstalled
#   sklearn 1.6.x is API-compatible for everything we touch (TfidfVectorizer,
#   LinearSVC, CalibratedClassifierCV, IsotonicRegression, train_test_split). Not
#   pinning avoids breaking umap-learn and hdbscan, which Colab also preinstalls.
import os, sys, subprocess

REPO_DIR = '/content/HybridGuard'
SRC      = f'{REPO_DIR}/src'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://github.com/ShaikhaTheGreen/HybridGuard.git', REPO_DIR],
                   check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

if SRC not in sys.path:
    sys.path.insert(0, SRC)

%pip install -q transformers datasets accelerate sentence-transformers tqdm regex

# Smoke test — both the canonicalization library and the GPU should be reachable.
import torch
from hybridguard import canonicalize, __version__ as HG_VERSION

assert torch.cuda.is_available(), 'No GPU detected. Switch to A100 in Runtime → Change runtime type.'
print(f'GPU         : {torch.cuda.get_device_name(0)}')
print(f'hybridguard : {HG_VERSION}')
_demo = canonicalize('Ｉｇｎｏｒｅ\u200b previous instructions')
assert 'Ignore previous instructions' in _demo.canonical, 'Canonicalization sanity check failed.'
print(f'canonicalize: {_demo.canonical!r}  trace={_demo.trace}')

In [ ]:
# 1.3 — Configure paths and verify Drive layout
from pathlib import Path
import datetime as _dt

DRIVE_ROOT = '/content/drive/MyDrive/HybridGuard'
RUN_ID     = f'revision_addons_{_dt.datetime.now().strftime("%Y%m%d_%H%M")}'

assert os.path.isdir('/content/drive/MyDrive'), \
    'Drive is not mounted. Run cell 1.1 first.'
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

DRIVE_OUT = Path(DRIVE_ROOT) / 'revision_addons' / RUN_ID
WS1_DIR   = DRIVE_OUT / 'paper' / 'paper_v2_extract' / 'ws1_universal'
ECE_DIR   = DRIVE_OUT / 'paper' / 'paper_v2_extract' / 'evaluation' / 'in_domain'
WS1_DIR.mkdir(parents=True, exist_ok=True)
ECE_DIR.mkdir(parents=True, exist_ok=True)

OUT_WS1_CSV       = WS1_DIR / 'results_with_ci.csv'
OUT_WS1_PERSAMPLE = WS1_DIR / 'per_sample_scores.joblib'
OUT_ECE_CSV       = ECE_DIR / 'calibration_std_ece.csv'
OUT_VAL_PREDS     = ECE_DIR / 'val_predictions.joblib'

# HG checkpoints (Section 4 only). We try a few plausible Drive layouts and
# gracefully skip if none match — the standard-ECE table will then cover the
# four non-HG detectors only, which is fine.
HG_CHECKPOINT_GLOBS = [
    f'{DRIVE_ROOT}/runs/run_*/seed_*/HG_*/*.joblib',
    f'{DRIVE_ROOT}/runs/*/seed_*/HG_*/*.joblib',
    f'{DRIVE_ROOT}/checkpoints/*HG_*.joblib',
]

print(f'Run ID       : {RUN_ID}')
print(f'Drive output : {DRIVE_OUT}')
print(f'  → A1 CSV   : {OUT_WS1_CSV.relative_to(Path(DRIVE_ROOT))}')
print(f'  → A2 CSV   : {OUT_ECE_CSV.relative_to(Path(DRIVE_ROOT))}')

## 2. Data and shared utilities

In [ ]:
# 2.1 — Load the xTRam1 prompt-injection benchmark.
#
# We adapt to whatever split structure and n the current dataset ships. The
# manuscript's Table 9 was generated on n=1625; if the current snapshot has
# a different n, the bootstrap CIs we compute below will be valid for that
# new n, and Table 9's caption + point estimates need to be updated to match.
from datasets import load_dataset
import numpy as np

ds = load_dataset('xTRam1/safe-guard-prompt-injection')

if 'test' in ds:
    test_split = ds['test']
    test_origin = "ds['test']"
elif 'validation' in ds:
    test_split = ds['validation']
    test_origin = "ds['validation']"
else:
    full = ds['train']
    test_split = full.train_test_split(test_size=0.10, seed=42, stratify_by_column='label')['test']
    test_origin = "derived from ds['train'] (10% stratified, seed=42)"

train_split = ds['train']  if 'train' in ds else None
y_test = np.asarray(test_split['label']).astype(int)
x_test = list(test_split['text'])
n_test = len(y_test)
n_pos  = int((y_test == 1).sum())
n_neg  = int((y_test == 0).sum())

print(f'Test source : {test_origin}')
print(f'Test set    : n={n_test}  (positives={n_pos}, negatives={n_neg})')
if n_test != 1625:
    print()
    print(f'⚠  Manuscript Table 9 was generated on n=1625; the current snapshot has n={n_test}.')
    print(f'   The bootstrap CIs we compute below will be for n={n_test}; remember to:')
    print(f'   (a) replace Table 9\'s point estimates with the new ones, and')
    print(f'   (b) update the caption and §5.11 prose from n=1625 to n={n_test}.')

In [ ]:
# 2.2 — Metric utilities (used throughout)
from sklearn.metrics import roc_auc_score

BOOT_SEED  = 2026
N_BOOT     = 1000
FPR_TARGET = 0.01

def recall_at_fpr(y_true, y_score, fpr_target=FPR_TARGET):
    y = np.asarray(y_true).astype(int); s = np.asarray(y_score).astype(float)
    n_neg, n_pos = int((y == 0).sum()), int((y == 1).sum())
    if n_neg == 0 or n_pos == 0:
        return float('nan')
    order = np.argsort(-s)
    fp_cum = np.cumsum(y[order] == 0)
    fpr_cum = fp_cum / n_neg
    eligible = np.where(fpr_cum <= fpr_target)[0]
    if len(eligible) == 0:
        return 0.0
    tau = s[order][eligible[-1]]
    return float(((s >= tau) & (y == 1)).sum() / n_pos)

def auroc(y_true, y_score):
    y = np.asarray(y_true)
    if len(np.unique(y)) < 2:
        return float('nan')
    return float(roc_auc_score(y, y_score))

def bootstrap_ci(y_true, y_score, n_boot=N_BOOT, rng=None):
    rng = rng or np.random.default_rng(BOOT_SEED)
    y = np.asarray(y_true).astype(int); s = np.asarray(y_score).astype(float)
    n = len(y)
    pt_a = auroc(y, s)
    pt_r = recall_at_fpr(y, s)
    a_b, r_b = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, sb = y[idx], s[idx]
        if len(np.unique(yb)) < 2:
            continue
        a_b.append(auroc(yb, sb))
        r_b.append(recall_at_fpr(yb, sb))
    return {
        'auroc'             : pt_a,
        'auroc_lo'          : float(np.percentile(a_b, 2.5)),
        'auroc_hi'          : float(np.percentile(a_b, 97.5)),
        'recall_at_1pctfpr' : pt_r,
        'recall_lo'         : float(np.percentile(r_b, 2.5)),
        'recall_hi'         : float(np.percentile(r_b, 97.5)),
    }

def ece_confidence(y_true, p_pos, n_bins=15):
    """Standard confidence-based ECE: bin on confidence = max(p, 1-p)."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos).astype(float)
    pred = (p >= 0.5).astype(int)
    conf = np.where(pred == 1, p, 1.0 - p)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx  = np.digitize(conf, bins[1:-1])
    n    = len(y)
    ece  = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0: continue
        ece += (m.sum() / n) * abs(float((pred[m] == y[m]).mean()) - float(conf[m].mean()))
    return float(ece)

def brier(y_true, p_pos):
    return float(((np.asarray(p_pos).astype(float) - np.asarray(y_true).astype(int)) ** 2).mean())

print(f'Utilities ready · N_BOOT={N_BOOT} · FPR target={FPR_TARGET}')

## 3. Section A1 — Bootstrap CIs for Table 9 (universal canonicalization recovery)

Four detectors × three conditions × 1000 bootstrap resamples. For each detector × condition we score the test set once, then resample 1000× with replacement to compute 95% CIs on AUROC and Recall@1%FPR.

Conditions:
* `clean` — original test text
* `perturbed` — Cyrillic-confusable homoglyph substitution applied
* `perturbed + canonical` — same perturbation, then our canonicalization front-end as a black-box pre-processor

In [ ]:
# 3.1 — Build the three text variants for each test sample
from hybridguard import canonicalize as _hg_canon

CYRILLIC_HOMOGLYPHS = {
    'a':'а','c':'с','e':'е','o':'о','p':'р','x':'х','y':'у',
    'A':'А','B':'В','C':'С','E':'Е','H':'Н','K':'К','M':'М',
    'O':'О','P':'Р','T':'Т','X':'Х','Y':'У',
}
def perturb_homoglyph(s: str) -> str:
    return ''.join(CYRILLIC_HOMOGLYPHS.get(c, c) for c in s)
def apply_canonicalize(s: str) -> str:
    return _hg_canon(s).canonical

x_clean               = list(x_test)
x_perturbed           = [perturb_homoglyph(s) for s in x_test]
x_perturbed_canonical = [apply_canonicalize(s) for s in x_perturbed]

i = next((i for i, y in enumerate(y_test) if y == 1), 0)
print('Sample (attack):')
print(f'  clean    : {x_clean[i][:100]}')
print(f'  perturbed: {x_perturbed[i][:100]}')
print(f'  + canon  : {x_perturbed_canonical[i][:100]}')

In [ ]:
# 3.2 — Detector 1: regex heuristic (deterministic, no GPU)
import re
REGEX_PATTERNS = [
    r'(?i)\bignore (all|previous|prior|the above) (instructions?|prompts?|rules?)\b',
    r'(?i)\b(disregard|forget|override) (all|previous|prior|earlier|the above) (instructions?|rules?)\b',
    r'(?i)\b(you are|act as|pretend to be) (now |an? )(?:dan|developer mode|jailbroken|unrestricted)\b',
    r'(?i)\b(reveal|show|print|display) (the |your )?(system prompt|hidden|secret) (instructions?|context)?\b',
    r'(?i)\bdo anything now\b',
    r'(?i)\b(begin|start|enter) (developer|admin|root|god) mode\b',
    r'(?i)\bnew (instructions?|directives?|persona)[:.]',
]
REGEX_COMPILED = [re.compile(p) for p in REGEX_PATTERNS]

def regex_score(texts):
    n_p = len(REGEX_COMPILED)
    return np.array([sum(1 for p in REGEX_COMPILED if p.search(t)) / n_p for t in texts])

scores_regex = {
    'clean':                regex_score(x_clean),
    'perturbed':            regex_score(x_perturbed),
    'perturbed_canonical':  regex_score(x_perturbed_canonical),
}
for cond, s in scores_regex.items():
    print(f'regex      | {cond:22s} | AUROC={auroc(y_test, s):.4f}  R@1%FPR={recall_at_fpr(y_test, s):.4f}')

In [ ]:
# 3.3 — Detector 2: TF-IDF + LinearSVM (trained fresh on the dataset's train split)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

assert train_split is not None, "This dataset has no 'train' split — cannot train TF-IDF baseline."
train_texts  = list(train_split['text'])
train_labels = np.asarray(train_split['label']).astype(int)
print(f'Training TF-IDF + LinearSVM on n={len(train_texts)} samples ...')

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50_000)
X_train = tfidf.fit_transform(train_texts)
svm = CalibratedClassifierCV(estimator=LinearSVC(C=1.0, max_iter=5000), cv=3)
svm.fit(X_train, train_labels)

def tfidf_svm_score(texts):
    return svm.predict_proba(tfidf.transform(texts))[:, 1]

scores_tfidf = {
    'clean':                tfidf_svm_score(x_clean),
    'perturbed':            tfidf_svm_score(x_perturbed),
    'perturbed_canonical':  tfidf_svm_score(x_perturbed_canonical),
}
for cond, s in scores_tfidf.items():
    print(f'tfidf      | {cond:22s} | AUROC={auroc(y_test, s):.4f}  R@1%FPR={recall_at_fpr(y_test, s):.4f}')

In [ ]:
# 3.4 — Detector 3: InjecGuard (HuggingFace)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_INJECGUARD = 'leolee99/InjecGuard'

ig_tok = AutoTokenizer.from_pretrained(MODEL_INJECGUARD)
ig_mod = AutoModelForSequenceClassification.from_pretrained(MODEL_INJECGUARD).to(device).eval()

@torch.inference_mode()
def injecguard_score(texts, batch_size=32):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc='InjecGuard'):
        enc = ig_tok(texts[i:i+batch_size], padding=True, truncation=True,
                     max_length=512, return_tensors='pt').to(device)
        out.append(torch.softmax(ig_mod(**enc).logits, dim=-1)[:, -1].cpu().numpy())
    return np.concatenate(out)

scores_injecguard = {
    'clean':                injecguard_score(x_clean),
    'perturbed':            injecguard_score(x_perturbed),
    'perturbed_canonical':  injecguard_score(x_perturbed_canonical),
}
for cond, s in scores_injecguard.items():
    print(f'InjecGuard | {cond:22s} | AUROC={auroc(y_test, s):.4f}  R@1%FPR={recall_at_fpr(y_test, s):.4f}')

del ig_mod
torch.cuda.empty_cache()

In [ ]:
# 3.5 — Detector 4: DeBERTa-v3 (protectai)
MODEL_DEBERTA = 'protectai/deberta-v3-base-prompt-injection-v2'

db_tok = AutoTokenizer.from_pretrained(MODEL_DEBERTA)
db_mod = AutoModelForSequenceClassification.from_pretrained(MODEL_DEBERTA).to(device).eval()

@torch.inference_mode()
def deberta_score(texts, batch_size=32):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc='DeBERTa-v3'):
        enc = db_tok(texts[i:i+batch_size], padding=True, truncation=True,
                     max_length=512, return_tensors='pt').to(device)
        out.append(torch.softmax(db_mod(**enc).logits, dim=-1)[:, -1].cpu().numpy())
    return np.concatenate(out)

scores_deberta = {
    'clean':                deberta_score(x_clean),
    'perturbed':            deberta_score(x_perturbed),
    'perturbed_canonical':  deberta_score(x_perturbed_canonical),
}
for cond, s in scores_deberta.items():
    print(f'DeBERTa-v3 | {cond:22s} | AUROC={auroc(y_test, s):.4f}  R@1%FPR={recall_at_fpr(y_test, s):.4f}')

del db_mod
torch.cuda.empty_cache()

In [ ]:
# 3.6 — Bootstrap CIs across all (detector × condition) and persist results
import joblib
import pandas as pd

all_scores = {
    'regex_heuristic': scores_regex,
    'tfidf_linearsvm': scores_tfidf,
    'InjecGuard':      scores_injecguard,
    'DeBERTa-v3':      scores_deberta,
}

rng = np.random.default_rng(BOOT_SEED)
rows = []
for detector, by_cond in all_scores.items():
    for condition, s in by_cond.items():
        rows.append({'detector': detector, 'condition': condition,
                     **bootstrap_ci(y_test, s, N_BOOT, rng)})

ws1_ci = pd.DataFrame(rows)
ws1_ci['n_test'] = n_test
ws1_ci.to_csv(OUT_WS1_CSV, index=False)
joblib.dump({'y_test': y_test, 'all_scores': all_scores}, OUT_WS1_PERSAMPLE)

ws1_round = ws1_ci.copy()
for c in ['auroc', 'auroc_lo', 'auroc_hi', 'recall_at_1pctfpr', 'recall_lo', 'recall_hi']:
    ws1_round[c] = ws1_round[c].round(3)
print(ws1_round.drop(columns=['n_test']).to_string(index=False))
print()
print(f'Saved:')
print(f'  {OUT_WS1_CSV}')
print(f'  {OUT_WS1_PERSAMPLE}')

# Manuscript-ready summary
def fmt_ci(pt, lo, hi):
    return f'{pt:.3f}\\,[{lo:.3f},\\,{hi:.3f}]'

print(f'\n=== Table 9 (manuscript-ready, n={n_test}) ===')
for det in ['regex_heuristic', 'tfidf_linearsvm', 'InjecGuard', 'DeBERTa-v3']:
    if det not in ws1_ci.detector.values:
        continue
    sub = ws1_ci[ws1_ci.detector == det].set_index('condition')
    print(f'  {det}:')
    for cond in ['clean', 'perturbed', 'perturbed_canonical']:
        if cond in sub.index:
            r = sub.loc[cond]
            print(f'    {cond:22s}  AUROC = {fmt_ci(r.auroc, r.auroc_lo, r.auroc_hi)}'
                  f'   R@1%FPR = {fmt_ci(r.recall_at_1pctfpr, r.recall_lo, r.recall_hi)}')

## 4. Section A2 — Standard confidence-based ECE for Table 5

We need validation-set probability outputs (raw and post-hoc-calibrated) for each model. The notebook scores the four non-HG detectors fresh, then attempts to load HybridGuard variant checkpoints from Drive. If checkpoints aren't present, the HG variants are gracefully skipped — Table 5 will then have standard-ECE values for the four non-HG detectors only, and the HG-variant rows can be filled in from a future orchestrator run.

In [ ]:
# 4.1 — Build a deterministic validation split from train (10% stratified, seed=42).
#
# Uses HuggingFace Dataset.select() with explicit Python ints — newer `datasets`
# releases reject numpy.int64 in Dataset.__getitem__, and .select() is the
# idiomatic subset-by-indices API regardless. This val split is independent of
# whatever split the orchestrator persisted; if exact reproducibility against the
# orchestrator's val set is required, swap in the saved split-indices file.
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(train_split)),
    test_size=0.10, random_state=42,
    stratify=np.asarray(train_split['label']),
)

val_subset = train_split.select([int(i) for i in val_idx])
x_val = list(val_subset['text'])
y_val = np.asarray(val_subset['label']).astype(int)

print(f'Validation split: n={len(y_val)}  '
      f'(positives={(y_val==1).sum()}, negatives={(y_val==0).sum()})')

In [ ]:
# 4.2 — Best-of-three post-hoc calibration: temperature / Platt / isotonic, selected by val ECE.
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit

EPS = 1e-7

def fit_temperature(p_val, y_val):
    z = logit(np.clip(p_val, EPS, 1 - EPS))
    def nll(T):
        if T <= 0:
            return 1e9
        p = expit(z / T)
        return -(y_val * np.log(p + EPS) + (1 - y_val) * np.log(1 - p + EPS)).mean()
    res = minimize_scalar(nll, bounds=(0.05, 10.0), method='bounded')
    T = float(res.x)
    return (lambda p: expit(logit(np.clip(np.asarray(p), EPS, 1 - EPS)) / T))

def fit_platt(p_val, y_val):
    z = logit(np.clip(p_val, EPS, 1 - EPS)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6).fit(z, y_val)
    return (lambda p: lr.predict_proba(
        logit(np.clip(np.asarray(p), EPS, 1 - EPS)).reshape(-1, 1))[:, 1])

def fit_isotonic(p_val, y_val):
    iso = IsotonicRegression(out_of_bounds='clip').fit(p_val, y_val)
    return (lambda p: iso.predict(np.asarray(p)))

def best_calibration(p_val, y_val):
    candidates = {
        'temperature': fit_temperature(p_val, y_val),
        'platt':       fit_platt(p_val, y_val),
        'isotonic':    fit_isotonic(p_val, y_val),
    }
    scored = sorted(
        [(ece_confidence(y_val, fn(p_val)), name, fn) for name, fn in candidates.items()],
        key=lambda t: t[0],
    )
    return scored[0][2], scored[0][1]   # (apply_fn, method_name)

print('Calibration helpers ready.')

In [ ]:
# 4.3 — Score the validation set under each detector (raw + post-hoc-calibrated)
val_predictions = {}    # detector -> {'y', 'p_raw', 'p_cal', 'cal_method'}

# 4.3a Regex (deterministic; calibration is a no-op)
p_raw = regex_score(x_val)
val_predictions['Regex heuristic'] = {
    'y': y_val, 'p_raw': p_raw, 'p_cal': p_raw, 'cal_method': '—',
}

# 4.3b TF-IDF + LinearSVM
p_raw = tfidf_svm_score(x_val)
apply_cal, name = best_calibration(p_raw, y_val)
val_predictions['TF-IDF + LinearSVM'] = {
    'y': y_val, 'p_raw': p_raw, 'p_cal': apply_cal(p_raw), 'cal_method': name,
}

# 4.3c InjecGuard
ig_tok = AutoTokenizer.from_pretrained(MODEL_INJECGUARD)
ig_mod = AutoModelForSequenceClassification.from_pretrained(MODEL_INJECGUARD).to(device).eval()
@torch.inference_mode()
def _ig_v(texts, bs=32):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc='InjecGuard val'):
        enc = ig_tok(texts[i:i+bs], padding=True, truncation=True,
                     max_length=512, return_tensors='pt').to(device)
        out.append(torch.softmax(ig_mod(**enc).logits, dim=-1)[:, -1].cpu().numpy())
    return np.concatenate(out)
p_raw = _ig_v(x_val)
apply_cal, name = best_calibration(p_raw, y_val)
val_predictions['SOTA: InjecGuard'] = {
    'y': y_val, 'p_raw': p_raw, 'p_cal': apply_cal(p_raw), 'cal_method': name,
}
del ig_mod; torch.cuda.empty_cache()

# 4.3d DeBERTa-v3
db_tok = AutoTokenizer.from_pretrained(MODEL_DEBERTA)
db_mod = AutoModelForSequenceClassification.from_pretrained(MODEL_DEBERTA).to(device).eval()
@torch.inference_mode()
def _db_v(texts, bs=32):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc='DeBERTa val'):
        enc = db_tok(texts[i:i+bs], padding=True, truncation=True,
                     max_length=512, return_tensors='pt').to(device)
        out.append(torch.softmax(db_mod(**enc).logits, dim=-1)[:, -1].cpu().numpy())
    return np.concatenate(out)
p_raw = _db_v(x_val)
apply_cal, name = best_calibration(p_raw, y_val)
val_predictions['SOTA: DeBERTa-v3'] = {
    'y': y_val, 'p_raw': p_raw, 'p_cal': apply_cal(p_raw), 'cal_method': name,
}
del db_mod; torch.cuda.empty_cache()

print()
print('Val predictions collected for:')
for k, v in val_predictions.items():
    print(f'  - {k:30s}  cal={v["cal_method"]}')

In [ ]:
# 4.4 — HybridGuard variants from Drive checkpoints (best-effort; gracefully skipped if absent)
import glob

found_ckpts = {}    # variant -> path
for variant in ['HG_MULTIFEAT', 'HG_CNN_TRANS', 'HG_ENSEMBLE', 'HG_RAV']:
    for pattern in HG_CHECKPOINT_GLOBS:
        matches = sorted(glob.glob(pattern))
        # restrict to entries whose path mentions this variant
        matches = [m for m in matches if variant in m]
        if matches:
            found_ckpts[variant] = matches[-1]    # most recent
            break

if not found_ckpts:
    print('No HybridGuard checkpoints found in Drive.')
    print('  Section 4 will produce standard-ECE for the 4 non-HG detectors only.')
    print('  This is fine — HG-variant rows in Table 5 can be added from a future orchestrator run.')
else:
    print(f'Found {len(found_ckpts)} HybridGuard checkpoint(s):')
    for v, p in found_ckpts.items():
        print(f'  - {v}: {p}')

for variant, ckpt_path in found_ckpts.items():
    try:
        bundle = joblib.load(ckpt_path)
        # The orchestrator's bundle layout varies — handle a few cases:
        if isinstance(bundle, dict) and callable(bundle.get('score_texts')):
            p_raw = np.asarray(bundle['score_texts'](x_val))
        elif isinstance(bundle, dict) and 'model' in bundle and hasattr(bundle['model'], 'predict_proba'):
            p_raw = np.asarray(bundle['model'].predict_proba(x_val))[:, 1]
        elif hasattr(bundle, 'predict_proba'):
            p_raw = np.asarray(bundle.predict_proba(x_val))[:, 1]
        else:
            print(f'  [skip] {variant}: bundle does not expose a predict_proba interface')
            continue
        apply_cal, name = best_calibration(p_raw, y_val)
        # Use the bare variant name as the dict key (no LaTeX escaping at the data layer).
        val_predictions[variant] = {
            'y': y_val, 'p_raw': p_raw, 'p_cal': apply_cal(p_raw), 'cal_method': name,
        }
        print(f'  loaded {variant} | cal={name}')
    except Exception as e:
        print(f'  [skip] {variant}: {type(e).__name__}: {e}')

In [ ]:
# 4.5 — Compute standard ECE per detector (raw + calibrated) and Brier on val
rows = []
for detector, pred in val_predictions.items():
    rows.append({
        'detector':              detector,
        'val_ece_raw_std':       ece_confidence(pred['y'], pred['p_raw']),
        'val_ece_cal_std':       ece_confidence(pred['y'], pred['p_cal']),
        'val_brier_raw':         brier(pred['y'], pred['p_raw']),
        'val_brier_cal':         brier(pred['y'], pred['p_cal']),
        'cal_method':            pred.get('cal_method', '—'),
    })

ece_df = pd.DataFrame(rows)
ece_df.to_csv(OUT_ECE_CSV, index=False)
joblib.dump(val_predictions, OUT_VAL_PREDS)

ece_round = ece_df.copy()
for c in ['val_ece_raw_std', 'val_ece_cal_std', 'val_brier_raw', 'val_brier_cal']:
    ece_round[c] = ece_round[c].round(4)
print('=== Standard confidence-based ECE (manuscript Table 5) ===\n')
print(ece_round.to_string(index=False))
print()
print(f'Saved:')
print(f'  {OUT_ECE_CSV}')
print(f'  {OUT_VAL_PREDS}')

## 5. Bundle outputs and trigger download

This cell is idempotent — re-running it just rebuilds the zip from whatever's currently in `DRIVE_OUT`.

In [ ]:
# 5.1 — Build a README, zip everything, save the zip to Drive, attempt browser download
import shutil

README_TEXT = f"""HybridGuard - Revision Addons
==============================
Run ID:    {RUN_ID}
Generated: {_dt.datetime.now().isoformat(timespec='seconds')}
Test n:    {n_test} (xTRam1 current snapshot)

WHAT'S IN THIS ZIP
------------------
paper/paper_v2_extract/ws1_universal/
  results_with_ci.csv         Bootstrap 95% CIs for Table 9 (universal recovery)
  per_sample_scores.joblib    Per-sample scores per (detector, condition); audit trail

paper/paper_v2_extract/evaluation/in_domain/
  calibration_std_ece.csv     Standard confidence-based ECE for Table 5
  val_predictions.joblib      Val-set raw / calibrated probabilities per detector

WHERE TO PUT IT
---------------
Unzip into the HybridGuard root folder on your Mac:
    /Users/<you>/.../OneDrive .../HybridGuard/
The zip preserves the manuscript-aligned subfolder layout, so the CSVs land
directly under paper/paper_v2_extract/ where the manuscript expects them.

Then:
  1. cd dashboard && python3 dashboard.py    -- the new Universal Defense
                                                tab + Standard ECE row appear.
  2. Edit paper/main.tex per docs/REVISION_QUEUE.md sections A1 and A2.
  3. Commit and push from your Mac terminal.

DRIVE COPY
----------
A persistent backup of these artifacts is at:
    {DRIVE_OUT}
If the browser download missed (e.g., tab closed during background execution),
re-download from there - no need to re-run the notebook.
"""
(DRIVE_OUT / 'README.txt').write_text(README_TEXT)

zip_stem  = f'/content/HybridGuard_RevisionAddons_{RUN_ID}'
zip_local = shutil.make_archive(zip_stem, 'zip', root_dir=str(DRIVE_OUT))
zip_drive = Path(DRIVE_OUT).parent / f'HybridGuard_RevisionAddons_{RUN_ID}.zip'
shutil.copy(zip_local, zip_drive)

print(f'Local zip (ephemeral): {zip_local}')
print(f'Drive zip (persistent): {zip_drive}')
print()

try:
    from google.colab import files
    files.download(zip_local)
    print('Browser download triggered.')
    print('If it didn\'t appear (e.g., tab was hidden during background execution),')
    print('open Drive in a new tab and grab the zip from:')
    print(f'    {zip_drive}')
except Exception as e:
    print(f'Browser download skipped ({type(e).__name__}: {e}).')
    print('Use the Drive copy instead:')
    print(f'    {zip_drive}')

## 6. After download — local checklist

1. **Unzip into OneDrive's HybridGuard folder.** Drag-and-drop or `unzip ~/Downloads/HybridGuard_RevisionAddons_*.zip -d ~/.../HybridGuard/`. The zip's internal layout drops every file into the right place (`paper/paper_v2_extract/...`).
2. **Re-launch the dashboard.** `cd dashboard && python3 dashboard.py` — the new **🛡️ Universal Defense** tab and the **Standard ECE** section under Calibration appear automatically. The info banner at the top says `✓ Bootstrap CIs loaded · ✓ Standard ECE loaded`.
3. **Update `paper/main.tex`.** See `docs/REVISION_QUEUE.md` sections A1 and A2 for the exact LaTeX edits (Table 9 cells get `\,[lo,\,hi]` after each value; Table 5 ECE columns get the `_std` values; the two queued-for-revised-version sentences get deleted). If `n_test ≠ 1625` (it is now), also update Table 9's caption from `n=1625` to the actual `n` and the §5.11 prose accordingly.
4. **Commit and push from your Mac.** Colab cannot push back to GitHub, but your local Mac can. The two CSVs and the manuscript edits should land in one commit titled "Revision-pass: bootstrap CIs (Table 9) + standard ECE (Table 5)."